# Contrails2: max-cluster sequence, attributes, directed degree balance

Uses `contrails2-cluster.json`, `contrails2-attr.json`, `contrails2-network.json`:

1. At each time step, pick the cluster with the **largest member count** (ties: **smaller numeric cluster ID**).
2. Print the cluster ID sequence in time order.
3. For each ID, read `length`, `mass`, `temp` from attr; min-max each attribute separately to [0, 1] using global min/max from `contrails2-attr.json`, then print.
4. On the network edges, compute **in-degree minus out-degree** per cluster and print.

Source: https://github.com/nafiul-nipu/Contrails

In [4]:
import json
from collections import defaultdict
from pathlib import Path

# Resolve JSON location when cwd is repo root or Scripts/
_candidates = [Path.cwd(), Path.cwd() / "Scripts"]
SCRIPT_DIR = next((p for p in _candidates if (p / "contrails2-cluster.json").exists()), Path.cwd())


def load_json(name: str):
    with open(SCRIPT_DIR / name, encoding="utf-8") as f:
        return json.load(f)


cluster_by_time = load_json("contrails2-cluster.json")
attr_by_id = load_json("contrails2-attr.json")
edges = load_json("contrails2-network.json")

print("Data directory:", SCRIPT_DIR.resolve())

Data directory: /Users/siyuanzhao/Documents/GitHub/Loom_General/Scripts/Contrais


## 1. Largest cluster ID per time step

In [5]:
def max_size_cluster_id(at_t: dict) -> str:
    """at_t: cluster_id -> [member ids]; maximize size, then smaller cluster ID."""
    return max(at_t.items(), key=lambda kv: (len(kv[1]), -int(kv[0])))[0]


time_keys = sorted(cluster_by_time.keys(), key=float)
max_cluster_sequence = [max_size_cluster_id(cluster_by_time[t]) for t in time_keys]

print("Time keys (sorted):", time_keys)
print("Max-size cluster ID per time:", max_cluster_sequence)
print("IDs only:", ", ".join(max_cluster_sequence))

Time keys (sorted): ['0.1', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2']
Max-size cluster ID per time: ['2', '7', '15', '20', '23', '25', '27', '33', '36', '42', '48']
IDs only: 2, 7, 15, 20, 23, 25, 27, 33, 36, 42, 48


## 2. length / mass / temp from attr (same order), each attribute min-max to [0, 1] (global range from attr file)

In [6]:
ATTR_KEYS = ("length", "mass", "temp")

rows = []
for t, cid in zip(time_keys, max_cluster_sequence):
    a = attr_by_id[cid]
    rows.append({"time": t, "cluster_id": cid, "length": a["length"], "mass": a["mass"], "temp": a["temp"]})


def global_min_max(attr: dict, key: str) -> tuple[float, float]:
    vals = [float(rec[key]) for rec in attr.values()]
    return min(vals), max(vals)


def to_01(x: float, lo: float, hi: float) -> float:
    if hi == lo:
        return 0.0
    return (x - lo) / (hi - lo)


ranges = {k: global_min_max(attr_by_id, k) for k in ATTR_KEYS}
for k, (lo, hi) in ranges.items():
    print(f"{k}: global min={lo}, max={hi}")

for r in rows:
    for k in ATTR_KEYS:
        lo, hi = ranges[k]
        r[f"{k}_01"] = to_01(float(r[k]), lo, hi)

for r in rows:
    print(
        f"t={r['time']}\t id={r['cluster_id']}\t length_01={r['length_01']:.6f}\t mass_01={r['mass_01']:.6f}\t temp_01={r['temp_01']:.6f}"
    )

try:
    import pandas as pd
    display(
        pd.DataFrame(rows)[
            ["time", "cluster_id", "length", "mass", "temp", "length_01", "mass_01", "temp_01"]
        ]
    )
except ImportError:
    pass

length: global min=0.106, max=13.201
mass: global min=1.8333040683396903e-05, max=0.03074515243104493
temp: global min=229.674, max=388.56
t=0.1	 id=2	 length_01=0.827721	 mass_01=0.905690	 temp_01=0.081329
t=0.11	 id=7	 length_01=0.725544	 mass_01=0.921553	 temp_01=0.066614
t=0.12	 id=15	 length_01=0.943032	 mass_01=0.980551	 temp_01=0.084929
t=0.13	 id=20	 length_01=0.909355	 mass_01=0.982787	 temp_01=0.082449
t=0.14	 id=23	 length_01=0.984345	 mass_01=1.000000	 temp_01=0.088038
t=0.15	 id=25	 length_01=1.000000	 mass_01=0.980930	 temp_01=0.092739
t=0.16	 id=27	 length_01=0.845743	 mass_01=0.951410	 temp_01=0.081304
t=0.17	 id=33	 length_01=0.931042	 mass_01=0.989401	 temp_01=0.088145
t=0.18	 id=36	 length_01=0.877052	 mass_01=0.960600	 temp_01=0.078899
t=0.19	 id=42	 length_01=0.734937	 mass_01=0.884586	 temp_01=0.073638
t=0.2	 id=48	 length_01=0.952348	 mass_01=0.956091	 temp_01=0.084507


,time,cluster_id,length,mass,temp,length_01,mass_01,temp_01
0,0.1,2,10.945,0.027847,242.596,0.827721,0.905690,0.081329
1,0.11,7,9.607,0.028335,240.258,0.725544,0.921553,0.066614
2,0.12,15,12.455,0.030148,243.168,0.943032,0.980551,0.084929
3,0.13,20,12.014,0.030216,242.774,0.909355,0.982787,0.082449
4,0.14,23,12.996,0.030745,243.662,0.984345,1.000000,0.088038
5,0.15,25,13.201,0.030159,244.409,1.000000,0.980930,0.092739
6,0.16,27,11.181,0.029252,242.592,0.845743,0.951410,0.081304
7,0.17,33,12.298,0.030419,243.679,0.931042,0.989401,0.088145
8,0.18,36,11.591,0.029535,242.210,0.877052,0.960600,0.078899
9,0.19,42,9.730,0.027199,241.374,0.734937,0.884586,0.073638


## 3. In-degree minus out-degree (network)

In [7]:
in_degree = defaultdict(int)
out_degree = defaultdict(int)
for e in edges:
    s, t = e["source"], e["target"]
    out_degree[s] += 1
    in_degree[t] += 1

print("time\tcluster_id\tin_deg\tout_deg\tin_minus_out")
for t, cid in zip(time_keys, max_cluster_sequence):
    nid = int(cid)
    indeg = in_degree[nid]
    outdeg = out_degree[nid]
    diff = indeg - outdeg
    print(f"{t}\t {cid}\t {indeg}\t {outdeg}\t {diff}")

time	cluster_id	in_deg	out_deg	in_minus_out
0.1	 2	 0	 2	 -2
0.11	 7	 5	 2	 3
0.12	 15	 8	 2	 6
0.13	 20	 5	 1	 4
0.14	 23	 3	 2	 1
0.15	 25	 1	 2	 -1
0.16	 27	 2	 2	 0
0.17	 33	 5	 2	 3
0.18	 36	 2	 2	 0
0.19	 42	 4	 2	 2
0.2	 48	 6	 0	 6


In [ ]:
import math
import json
from pathlib import Path

# Recompute degree balance so this cell is runnable on its own after data-load cells.
in_degree = defaultdict(int)
out_degree = defaultdict(int)
for e in edges:
    s, t = int(e["source"]), int(e["target"])
    out_degree[s] += 1
    in_degree[t] += 1

times = [float(t) for t in time_keys]
nodes = [int(cid) for cid in max_cluster_sequence]
lower_values = [in_degree[n] - out_degree[n] for n in nodes]

def build_axis_order(values: list[int], step: int = 2) -> list[int]:
    lo, hi = min(values), max(values)
    start = step * math.floor(lo / step)
    end = step * math.ceil(hi / step)
    return list(range(start, end + 1, step))

glyph_json = {
    "paths": [
        {
            "label": "Contrail",
            "nodes": nodes,
            "times": times,
            "lower_values": lower_values,
        }
    ],
    "axis_order": build_axis_order(lower_values, step=2),
    "series_by_path": [
        {
            "path_index": 0,
            "series": [
                {
                    "id": "length",
                    "times": times,
                    "values": [round(float(r["length_01"]), 6) for r in rows],
                },
                {
                    "id": "mass",
                    "times": times,
                    "values": [round(float(r["mass_01"]), 6) for r in rows],
                },
            ],
        }
    ],
}

# Resolve output path for either repo-root or Scripts/* working directories.
_repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    SCRIPT_DIR,
    SCRIPT_DIR.parent,
    SCRIPT_DIR.parent.parent,
]
repo_root = next((p for p in _repo_candidates if (p / "Frontend" / "src" / "data").exists()), Path.cwd())
out_path = repo_root / "Frontend" / "src" / "data" / "contrails2MaxClusterGlyph.json"
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(glyph_json, f, indent=2)

print("Wrote:", out_path.resolve())
print(json.dumps(glyph_json, indent=2))

Wrote: /Users/siyuanzhao/Documents/GitHub/Loom_General/Frontend/src/contrails2MaxClusterGlyph.json
{
  "paths": [
    {
      "label": "contrail",
      "nodes": [
        2,
        7,
        15,
        20,
        23,
        25,
        27,
        33,
        36,
        42,
        48
      ],
      "times": [
        0.1,
        0.11,
        0.12,
        0.13,
        0.14,
        0.15,
        0.16,
        0.17,
        0.18,
        0.19,
        0.2
      ],
      "lower_values": [
        -2,
        3,
        6,
        4,
        1,
        -1,
        0,
        3,
        0,
        2,
        6
      ]
    }
  ],
  "axis_order": [
    -2,
    0,
    2,
    4,
    6
  ],
  "series_by_path": [
    {
      "path_index": 0,
      "series": [
        {
          "id": "length",
          "times": [
            0.1,
            0.11,
            0.12,
            0.13,
            0.14,
            0.15,
            0.16,
            0.17,
            0.18,
           